# 4. Model Training

## Objective
Train an ensemble of translation models for Akkadian to English translation.

### Approach:
Given the low-resource nature of this task (~1400 training samples), we use:
1. **Fine-tuned MarianMT** - A pre-trained NMT model fine-tuned on our data
2. **T5-base** - Text-to-text transfer transformer
3. **Ensemble** - Combine predictions from multiple models

The evaluation metric is the Geometric Mean of BLEU and chrF++ scores.

In [ ]:
# Parameters and Paths
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

OUTPUT_DIR = Path("output")
MODELS_DIR = OUTPUT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESSED_DIR = OUTPUT_DIR / "pre_processed_data"

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import translation libraries
from transformers import (
    MarianTokenizer, 
    MarianMTModel,
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset
import sacrebleu
from tqdm import tqdm

print("Libraries imported successfully")

## 4.1 Load Preprocessed Data

The data has been preprocessed by notebook 03, which combines:
- Original training data from `data/train.csv`
- Akkadian-English parallel corpus downloaded from Akkademia (ORACC RINAP data)

This combined dataset provides more training examples for the model.

In [ ]:
# Load preprocessed data
train_df = pd.read_csv(PREPROCESSED_DIR / "train_split.csv")
val_df = pd.read_csv(PREPROCESSED_DIR / "val_split.csv")

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Show data source breakdown if available
if 'source' in train_df.columns:
    print(f"\nData sources in training set:")
    print(train_df['source'].value_counts())

## 4.2 Create Datasets for Training

In [ ]:
# Create HuggingFace datasets
train_dataset = HFDataset.from_pandas(train_df[['transliteration_cleaned', 'translation_cleaned']])
val_dataset = HFDataset.from_pandas(val_df[['transliteration_cleaned', 'translation_cleaned']])

# Rename columns for consistency
train_dataset = train_dataset.rename_column('transliteration_cleaned', 'source')
train_dataset = train_dataset.rename_column('translation_cleaned', 'target')
val_dataset = val_dataset.rename_column('transliteration_cleaned', 'source')
val_dataset = val_dataset.rename_column('translation_cleaned', 'target')

print("Datasets created")
print(train_dataset)

## 4.3 Model 1: Fine-tuned MarianMT (Helsinki-NLP)

We start with a pre-trained MarianMT model and fine-tune it on our data. Since there's no pre-trained Akkadian model, we'll use a multilingual model and adapt it.

In [ ]:
# Since we don't have a pre-trained Akkadian->English model, we'll use
# a multilingual approach. We'll create a custom tokenizer for Akkadian.

# For this task, we'll train from scratch with a small transformer
# The key is to use subword tokenization that can handle Akkadian script

from transformers import MarianTokenizer, MarianMTModel

# Use a multilingual model that can be fine-tuned
# We'll use opus-mt-mul-en as a base (multi-language to English)
# But first, let's prepare the data format

# Prepare source/target format for MarianMT
def preprocess_function(examples):
    """Prepare examples for MarianMT"""
    # Add prefix for source language
    inputs = ["<unk> " + src for src in examples["source"]]
    targets = [tgt for tgt in examples["target"]]
    
    # Tokenize
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    
    # Tokenize targets
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing function defined")

## 4.4 Due to Resource Constraints - Simplified Training Approach

Given the computational constraints, we'll use a simplified but effective approach:
1. Train a sequence-to-sequence model with custom tokenization
2. Use the preprocessed data efficiently

In [ ]:
# Given the constraints, let's use a practical approach:
# - Use a small T5 model fine-tuned on our data
# - Use sentencepiece for better tokenization of Akkadian

# First, let's prepare the data in the right format

# For now, let's check what's the simplest working approach
# We'll use a smaller model that can train quickly

from transformers import T5Tokenizer, T5ForConditionalGeneration

# Use a small T5 model
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model = model.to(device)
print(f"Loaded {model_name}")

In [ ]:
# Prepare data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Tokenize datasets
def tokenize_t5(examples):
    """Tokenize for T5"""
    # Add prefix for translation task
    inputs = ["translate Akkadian to English: " + src for src in examples["source"]]
    targets = examples["target"]
    
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    
    # Tokenize targets
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(tokenize_t5, batched=True, remove_columns=train_dataset.column_names).with_format("torch")
val_tokenized = val_dataset.map(tokenize_t5, batched=True, remove_columns=val_dataset.column_names).with_format("torch")

print(f"Tokenized training samples: {len(train_tokenized)}")
print(f"Tokenized validation samples: {len(val_tokenized)}")

## 4.5 Training Configuration

In [ ]:
# Training arguments - optimized for our small dataset
training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "t5_akkadian"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    save_total_limit=2,
    fp16=False,  # Set to True if GPU supports it
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=0,
)

print("Training arguments configured")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer created")

## 4.6 Train the Model

In [ ]:
# Train the model
# Note: This may take a while depending on resources
print("Starting training...")
train_result = trainer.train()
print("Training complete!")

In [ ]:
# Save the model
model_save_path = MODELS_DIR / "t5_akkadian_translator"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

## 4.7 Evaluation Function

In [ ]:
# Create evaluation function
def compute_metrics(preds, labels):
    """Compute BLEU and chrF++ metrics"""
    # Decode predictions
    pred_strs = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Filter empty strings
    pred_strs = [p.strip() for p in pred_strs if p.strip()]
    label_strs = [l.strip() for l in label_strs if l.strip()]
    
    # Compute BLEU
    bleu = sacrebleu.corpus_bleu(pred_strs, [label_strs])
    
    # Compute chrF++
    chrf = sacrebleu.corpus_chrf(pred_strs, [label_strs])
    
    # Geometric mean
    geo_mean = np.sqrt(bleu.score * chrf.score)
    
    return {
        "bleu": bleu.score,
        "chrf": chrf.score,
        "geometric_mean": geo_mean
    }

print("Metrics function defined")

In [ ]:
# Evaluate on validation set
model.eval()

all_preds = []
all_labels = []

for batch in tqdm(DataLoader(val_tokenized, batch_size=8), desc="Evaluating"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels']
    
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128)
    
    all_preds.extend(outputs.cpu().numpy())
    all_labels.extend(labels)

metrics = compute_metrics(all_preds, all_labels)
print("\n=== Validation Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.2f}")

## 4.8 Training History and Loss Curves

In [ ]:
# Save training history
import matplotlib.pyplot as plt

history = train_result.history

# Plot loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'], label='Training Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

if 'eval_loss' in history:
    axes[1].plot(history['eval_loss'], label='Validation Loss', color='orange')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss')
    axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "images/training_loss.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Loss plot saved")

## 4.9 Save Training Summary

In [ ]:
# Save training summary
# Get data source info if available
data_sources = {}
if 'source' in train_df.columns:
    data_sources = train_df['source'].value_counts().to_dict()

training_summary = {
    "model_type": "T5-small",
    "model_name": "t5-small",
    "training_samples": len(train_tokenized),
    "validation_samples": len(val_tokenized),
    "num_epochs": 10,
    "learning_rate": 3e-4,
    "batch_size": 8,
    "data_sources": data_sources,
    "external_data": "Akkademia/ORACC RINAP parallel corpus included in training",
    "metrics": {
        "bleu": float(metrics['bleu']),
        "chrf": float(metrics['chrf']),
        "geometric_mean": float(metrics['geometric_mean'])
    },
    "model_path": str(model_save_path)
}

with open(MODELS_DIR / "training_summary.json", 'w') as f:
    json.dump(training_summary, f, indent=2)

print("Training summary saved:")
print(json.dumps(training_summary, indent=2))

## 4.10 List Saved Models

In [ ]:
# List all saved models
print("Saved models and files:")
for f in MODELS_DIR.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DIR)}: {f.stat().st_size / 1024:.1f} KB")

In [ ]:
# Verify notebook is valid JSON
notebook_path = Path("notebooks/04_model_training.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")